In [ ]:
import re
import pandas as pd
from collections import defaultdict

# ========== Step 1: Load FinancialPhraseBank ==========

def load_financial_phrasebank(file_path):
    with open(file_path, "r", encoding="latin-1") as file:
        lines = file.readlines()

    data = []
    for line in lines:
        line = line.strip()
        if '@' in line:
            try:
                sentence, label = line.rsplit('@', 1)
                sentence = sentence.strip().strip('"')
                label = label.strip().lower()
                if label in ['positive', 'negative', 'neutral']:
                    data.append({'sentence': sentence, 'label': label})
            except:
                continue

    df = pd.DataFrame(data)
    print(f"✅ Loaded FinancialPhraseBank: {len(df)} rows")
    print(df.head())
    return df


# ========== Step 2: Load NRC Emotion Lexicon ==========

def load_nrc_emotion_lexicon(file_path):
    emotion_dict = defaultdict(list)

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            word, emotion, association = line.strip().split('\t')
            if int(association) == 1:
                emotion_dict[word].append(emotion)

    print(f"✅ Loaded NRC Emotion Lexicon with {len(emotion_dict)} words.")
    return emotion_dict


# ========== Step 3: Save Results ==========

def save_outputs(df_fpb, emotion_dict):
    df_fpb.to_csv("parsed_financial_phrasebank.csv", index=False)
    print("✅ Saved FinancialPhraseBank to 'parsed_financial_phrasebank.csv'")

    # Optional: Save emotion_dict to a pickle file
    import pickle
    with open("nrc_emotion_dict.pkl", "wb") as f:
        pickle.dump(emotion_dict, f)
    print("✅ Saved NRC Emotion Lexicon to 'nrc_emotion_dict.pkl'")


# ========== MAIN ==========

if __name__ == "__main__":
    # Set file paths
    fpb_path = "C:/Users/DELL/Desktop/Capstone Project/FinancialPhraseBank-v1.0/Sentences_AllAgree.txt"
    nrc_path = "C:/Users/DELL/Desktop/Capstone Project/NRC-Emotion-Lexicon/NRC-Emotion-Lexicon-Wordlevel-v0.92.txt"

    # Load datasets
    df_fpb = load_financial_phrasebank(fpb_path)
    emotion_dict = load_nrc_emotion_lexicon(nrc_path)

    # Save parsed results
    save_outputs(df_fpb, emotion_dict)

In [1]:
import pandas as pd
import pickle
import nltk
from collections import defaultdict, Counter
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import string

# Download resources
nltk.download('punkt')
nltk.download('stopwords')

# Load saved data
df_fpb = pd.read_csv("parsed_financial_phrasebank.csv")

with open("nrc_emotion_dict.pkl", "rb") as f:
    emotion_dict = pickle.load(f)

# Basic text cleaning
stop_words = set(stopwords.words('english'))
punctuations = set(string.punctuation)

def preprocess(text):
    tokens = word_tokenize(text.lower())
    return [word for word in tokens if word not in stop_words and word not in punctuations]

# Tag each sentence with emotion counts
def tag_emotions(sentence):
    tokens = preprocess(sentence)
    emotion_counter = Counter()
    for word in tokens:
        if word in emotion_dict:
            emotion_counter.update(emotion_dict[word])
    return emotion_counter

# Apply tagging
emotion_data = df_fpb['sentence'].apply(tag_emotions)

# Extract all emotion columns
all_emotions = set()
for emo_count in emotion_data:
    all_emotions.update(emo_count.keys())

# Add emotion columns to DataFrame
for emotion in sorted(all_emotions):
    df_fpb[emotion] = emotion_data.apply(lambda x: x.get(emotion, 0))

# Save to new CSV
df_fpb.to_csv("financial_phrasebank_with_emotions.csv", index=False)
print("✅ Saved tagged dataset to 'financial_phrasebank_with_emotions.csv'")

# Preview
print(df_fpb.head())


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


PermissionError: [Errno 13] Permission denied: 'financial_phrasebank_with_emotions.csv'